[目录（Table of Contents）](./table_of_contents.ipynb)

# 最小二乘滤波（Least Squares Filters）

In [1]:
%matplotlib inline

In [2]:
#format the book
import book_format
book_format.set_style()

## 引言（Introduction）

**作者注**：本节从 g-h 滤波（g-h filter）一章中摘出，放在那里并不合适。本章尚未写完，目前还不适合阅读！

在章节开头，我使用 `numpy.polyfit()` 对体重测量数据拟合了一条直线。它通过「最小二乘拟合（least squared fit）」将 $n$ 次多项式拟合到数据上。这与 g-h 滤波有何不同？

嗯，这取决于具体情况。我们最终会学到，在特定条件下，卡尔曼滤波（Kalman filter）从最小二乘（least squares）拟合的角度来看是最优的。然而，`polyfit()` 是通过最小化下列公式，将多项式（而非任意曲线）拟合到数据上：

$$E = \sum_{j=0}^k |p(x_j) - y_j|^2$$

我假设体重以每天 1 磅的恒定速率增长，因此当我尝试拟合 $n=1$ 次（即直线）多项式时，结果与实际的体重增长非常接近。但当然，没有人会持续地只增重或只减重，体重会波动。对更长的数据序列使用 `polyfit()` 会得到较差的结果。相比之下，g-h 滤波会对速率变化做出反应——$h$ 项控制滤波器对这些变化响应的快慢。如果我们先增重、保持一段时间、再减重，滤波器会自动跟踪这种变化。除非增减过程能很好地用多项式表示，否则 `polyfit()` 无法做到这一点。

这种形式滤波的另一项优势是，即使数据能很好地用 $n$ 次多项式表示，它也是*递归（recursive）*的。也就是说，我们只需知道上一时刻的估计值和速率，就能计算当前时刻的估计值。相比之下，如果你深入查看 `polyfit()` 的实现，会发现它需要先收集全部数据才能给出结果。因此像 `polyfit()` 这样的算法并不适合实时数据滤波。在 20 世纪 60 年代卡尔曼滤波（Kalman filter）问世时，计算机速度很慢、内存极为有限，根本无法存储例如来自飞机惯性导航系统（inertial navigation system）的数千条读数，也无法在提供准确、及时导航信息所需的短时间内处理全部数据。


直到 20 世纪中叶，各种形式的最小二乘估计（Least Squares Estimation）被用于这类滤波。例如，NASA 的阿波罗（Apollo）计划设有地面网络，用于跟踪指令/服务舱（Command and Service Module, CSM）和登月舱（Lunar Module, LM）。他们采集数分钟内的测量数据，将其批量汇总，再缓慢地计算出结果。1960 年，NASA Ames 的 Stanley Schmidt 认识到 Rudolf Kalman 开创性论文的价值，并邀请他来到 Ames。Schmidt 将 Kalman 的工作应用于 CSM 和 LM 的机载导航系统，并将其称为「卡尔曼滤波（Kalman filter）」。[1] 此后不久，世界便转向这种更快、递归的滤波方法。

卡尔曼滤波（Kalman filter）只需存储上一次的估计值和少量相关参数，且只需相对较少的计算即可生成下一次估计。如今我们拥有大量内存和处理能力，这一优势已不那么突出，但在当时，卡尔曼滤波不仅因其数学性质而成为重大突破，还因为它（勉强）能在当时的硬件上运行。

这一主题远比本短讨论所暗示的更为深入。全书后续还将多次讨论这些话题。